In [81]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [82]:
import torch as th
import os
from fragnnet.runner import load_config, init_dataset
from fragnnet.utils.spec_utils import get_ints_transform_func, get_ints_untransform_func, batched_l1_normalize, batched_l2_normalize, batched_mf1000_normalize
from fragnnet.utils.misc_utils import print_shapes, safelog, scatter_reduce

In [83]:
# load base config
template_fp = "../config/template.yml"
# custom_fp = "../config/debug_um/debug_neims_auto_um.yml"
custom_fp = "../config/debug_m/debug_neims_m.yml"
config_d = load_config(template_fp, custom_fp)
config_d["ints_transform"] = "none"
config_d["spec_params"]["preprocess"] = False
config_d["mol_params"]["preprocess"] = False
for k in list(config_d.keys()):
    if k.endswith("_fp") or k.endswith("_dp"):
        if config_d[k] is not None:
            print(k)
            config_d[k] = os.path.join("..", config_d[k])

spec_fp
mol_fp
split_dp
frag_dp
magma_dp
ann_fp


In [84]:
# load dataset (without ints transform)
ds = init_dataset(config_d,splits=["val"])[0]
dl = th.utils.data.DataLoader(ds, batch_size=8, shuffle=False, collate_fn=ds.collate_fn)

In [85]:
# get ints transform/untransform functions
no_t = get_ints_transform_func("none")
no_ut = get_ints_untransform_func("none")
sqrt_t = get_ints_transform_func("sqrt")
sqrt_ut = get_ints_untransform_func("sqrt")
log_t = get_ints_transform_func("log10t3")
log_ut = get_ints_untransform_func("log10t3")
norm1 = batched_l1_normalize
norm2 = batched_l2_normalize
bsum = lambda x, b: scatter_reduce(x, b, "sum")
bmax = lambda x, b: scatter_reduce(x, b, "max")

In [86]:
# get a batch
b = next(iter(dl))
# print_shapes(b)
spec_ints = b["spec_ints"]
spec_batch_idxs = b["spec_batch_idxs"]
# spec_ints = batched_mf1000_normalize(spec_ints, spec_batch_idxs)
print(bsum(spec_ints, spec_batch_idxs))
print(bmax(spec_ints, spec_batch_idxs))

tensor([ 9619.0801,  8144.8706,  6690.2104,  5002.2100,  3665.1299,  4113.4297,
         8070.9897, 11646.1406])
tensor([1998.0000, 1047.0500,  999.0000, 1136.6600,  999.0000, 1998.0000,
        1998.0000, 2908.6899])


In [87]:
no_spec_t = norm(no_t(spec_ints), spec_batch_idxs)
no_spec_tl = safelog(no_spec_t)
no_spec_ut = no_ut(th.exp(no_spec_tl), spec_batch_idxs)
print(th.max(th.abs(spec_ints - no_spec_ut)))
print(th.max(th.abs(norm1(spec_ints, spec_batch_idxs) - norm1(no_spec_ut, spec_batch_idxs))))
print(th.max(th.abs(norm2(spec_ints, spec_batch_idxs) - norm2(no_spec_ut, spec_batch_idxs))))

tensor(1908.6899)
tensor(1.1921e-07)
tensor(1.1921e-07)


In [88]:
sqrt_spec_t = norm(sqrt_t(spec_ints), spec_batch_idxs)
sqrt_spec_tl = safelog(sqrt_spec_t)
sqrt_spec_ut = sqrt_ut(th.exp(sqrt_spec_tl), spec_batch_idxs)
print(th.max(th.abs(spec_ints - sqrt_spec_ut)))
print(th.max(th.abs(norm1(spec_ints, spec_batch_idxs) - norm1(sqrt_spec_ut, spec_batch_idxs))))
print(th.max(th.abs(norm2(spec_ints, spec_batch_idxs) - norm2(sqrt_spec_ut, spec_batch_idxs))))

tensor(1908.6899)
tensor(1.0431e-07)
tensor(2.3842e-07)


In [89]:
log_spec_t = norm(log_t(spec_ints), spec_batch_idxs)
log_spec_tl = safelog(log_spec_t)
log_spec_ut = log_ut(th.exp(log_spec_tl), spec_batch_idxs)
print(th.max(th.abs(spec_ints - log_spec_ut)))
print(th.max(th.abs(norm1(spec_ints, spec_batch_idxs) - norm1(log_spec_ut, spec_batch_idxs))))
print(th.max(th.abs(norm2(spec_ints, spec_batch_idxs) - norm2(log_spec_ut, spec_batch_idxs))))

tensor(1908.6904)
tensor(0.0671)
tensor(0.0612)
